# Inspect raw data — Nasdaq-100

Workflow step 1 (Data). Loads what `trading data-sync` downloaded into `data/raw/nasdaq-100/` and shows:

1. **Membership**: which stocks were in the index, and when
2. **Download manifest**: one row per series, with its bar count and date range
3. **One stock's bars**: timestamps, trading sessions, price chart
4. **Coverage**: does each series have bars for its whole membership period?

Read-only: nothing here writes to `data/`.

In [ ]:
import json
from datetime import date

import matplotlib.pyplot as plt
import polars as pl

from trading.data.api import load_membership, members_on
from trading.shared.storage import DATA_DIR

RAW = DATA_DIR / "raw" / "nasdaq-100"
BARS = RAW / "bars_20min"
NY = "America/New_York"
_ = pl.Config.set_tbl_rows(20)

## 1. Membership

One row per (ticker, membership period). `left` is null for current members. `series` is the download id: a ticker reused by a different company gets a year suffix (e.g. `SNDK.2016` is the old SanDisk).

In [ ]:
mem = load_membership()
print(f"{mem.height} periods, {mem['ticker'].n_unique()} tickers, {mem['series'].n_unique()} series")
mem.sort("joined", descending=True).head(10)

In [ ]:
# Number of index members on the 1st of each month (should stay around 100-107)
days = pl.date_range(date(2016, 1, 1), date.today(), "1mo", eager=True).to_list()
counts = [len(members_on(d)) for d in days]
plt.figure(figsize=(10, 3))
plt.plot(days, counts)
plt.title("Nasdaq-100 members over time")
plt.ylabel("stocks")
plt.show()

In [ ]:
# Stocks that left the index (these are the ones a "full history only" universe would miss)
mem.filter(pl.col("left").is_not_null()).sort("left", descending=True).head(15)

## 2. Download manifest

One entry per series: bar count and first/last bar. Short histories are usually recent IPOs or spin-offs.

In [ ]:
manifest = json.loads((RAW / "_manifest.json").read_text())
man = pl.DataFrame([{"series": k, **v} for k, v in manifest.items()]).with_columns(
    pl.col("first", "last").str.to_datetime(time_zone="UTC")
)
print(f"{man.height} series, {man['rows'].sum():,} bars in total")
man.select("series", "ticker", "rows", "first", "last").sort("rows").head(15)

## 3. One stock's bars

Timestamps are UTC bar starts. Alpaca returns pre-market, after-hours and overnight bars too. Regular hours are 09:30–16:00 New York time.

Bars start at :00, :20 and :40, so they don't line up with the 09:30 open: the 09:20 bar mixes 10 minutes of pre-market with the first 10 minutes of the session. Below, `regular` means the bar lies fully inside regular hours (09:40 … 15:40 starts, 19 bars per full day).

In [ ]:
SYMBOL = "AAPL"  # change to any series id, e.g. "SNDK.2016"

bars = pl.read_parquet(BARS / f"series={SYMBOL}.parquet").with_columns(
    ts_ny=pl.col("ts").dt.convert_time_zone(NY)
).with_columns(
    day=pl.col("ts_ny").dt.date(),
    minute=pl.col("ts_ny").dt.hour().cast(pl.Int32) * 60 + pl.col("ts_ny").dt.minute().cast(pl.Int32),
).with_columns(
    regular=(pl.col("minute") >= 9 * 60 + 30) & (pl.col("minute") < 16 * 60)
)
print(f"{SYMBOL}: {bars.height:,} bars, {bars['ts_ny'].min()} → {bars['ts_ny'].max()}")
bars.select("ts_ny", "open", "high", "low", "close", "volume", "trade_count", "vwap").tail(10)

In [ ]:
# How much of the data is regular hours vs extended / overnight
bars.group_by("regular").agg(
    bars=pl.len(), volume_share=pl.col("volume").sum()
).with_columns(pl.col("volume_share") / pl.col("volume_share").sum())

In [ ]:
# Regular-hours bars per day: 19 on a full day, fewer on half-days or days with missing bars
per_day = bars.filter("regular").group_by("day").agg(n=pl.len()).sort("day")
per_day["n"].value_counts().sort("count", descending=True).head(8)

In [ ]:
# Daily close (last regular-hours bar). Prices are split-adjusted, so no jumps should appear at splits.
daily = bars.filter("regular").group_by("day").agg(pl.col("close").last()).sort("day")
plt.figure(figsize=(10, 3))
plt.plot(daily["day"], daily["close"])
plt.title(f"{SYMBOL} daily close")
plt.yscale("log")
plt.show()

In [ ]:
# Largest bar-to-bar moves (incl. overnight gaps): check these look like real news, not data errors
bars.filter("regular").with_columns(ret=pl.col("close").pct_change()).sort(
    pl.col("ret").abs(), descending=True, nulls_last=True
).select("ts_ny", "open", "close", "ret", "volume").head(10)

## 4. Coverage: bars vs membership

For each membership period: when the index added the stock, and when its bars start. A stock whose bars start after `joined` (e.g. a new IPO) can only be traded from its first bar.

In [ ]:
cov = mem.join(man.select("series", "rows", "first", "last"), on="series", how="left").with_columns(
    first_bar=pl.col("first").dt.date(), last_bar=pl.col("last").dt.date()
).with_columns(
    missing_start_days=(pl.col("first_bar") - pl.max_horizontal("joined", pl.lit(date(2016, 1, 4)))).dt.total_days()
)
print("periods with no data:", cov.filter(pl.col("rows").is_null() | (pl.col("rows") == 0)).height)
cov.select("series", "joined", "left", "first_bar", "last_bar", "missing_start_days").sort(
    "missing_start_days", descending=True
).head(15)